# Module 06 — Convolutional Neural Networks
## 06-05: U-Net & Encoder-Decoder Architecture

**Objective:** Build Fully Convolutional Networks (FCN-32s, FCN-8s) and U-Net from scratch in PyTorch, understanding transposed convolution, bilinear upsampling, and skip connections for pixel-wise semantic segmentation.

**Prerequisites:** 06-01 (Conv2d, Pooling, Batch Norm), 06-02 (CNN Architectures), 06-05 (Residual Connections), 09-01 (Object Detection Foundations), 09-02 (Single-Stage Detector)

---
## Part 0 — Setup & Prerequisites

Semantic segmentation assigns a **class label to every pixel** in an image — unlike object detection which predicts bounding boxes, or image classification which predicts a single label per image. This is the foundation for applications like autonomous driving (lane/pedestrian detection), medical imaging (tumour delineation), and robotics scene understanding.

In this notebook we implement two landmark architectures:
- **FCN (Fully Convolutional Network):** The original "pixels-in, labels-out" network. We build both the FCN-32s (coarse, single upsampling) and FCN-8s (refined, skip-connection upsampling) variants.
- **U-Net:** An encoder-decoder architecture with symmetric skip connections at every resolution, originally designed for biomedical image segmentation but now widely used across vision tasks.

We train on a synthetic dataset of coloured shapes with $C=4$ classes: background + three shape types. Images are small ($64 \times 64$) so all training runs quickly on CPU.

**What we build:**
1. Synthetic segmentation dataset (800 train / 200 val samples)
2. Pixel-wise IoU metric from scratch
3. Transposed convolution vs bilinear upsampling comparison
4. FCN-32s and FCN-8s with skip connections
5. U-Net with encoder-decoder skip connections
6. Training all three models and comparing them qualitatively and quantitatively

**Prerequisites:** 06-01 (Conv2d, pooling, batch norm — see Module 06 for full treatment), 09-01 (anchor boxes / IoU concepts applied here to pixel level).

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import math
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
IMAGE_SIZE: int = 64          # Spatial resolution (H = W = 64)
NUM_CLASSES: int = 4          # Background (0) + Rectangle (1) + Circle (2) + Triangle (3)
NUM_TRAIN: int = 800          # Training samples
NUM_VAL: int = 200            # Validation samples
MAX_SHAPES: int = 4           # Max shapes per image
MIN_SHAPE_SIZE: int = 8       # Minimum bounding box dimension for a shape
MAX_SHAPE_SIZE: int = 20      # Maximum bounding box dimension for a shape

BATCH_SIZE: int = 32
NUM_EPOCHS: int = 8
LEARNING_RATE: float = 1e-3

# Colour palette for visualising segmentation masks (one colour per class)
PALETTE: list[tuple[float, float, float]] = [
    (0.15, 0.15, 0.15),   # Class 0: Background — dark grey
    (0.85, 0.25, 0.25),   # Class 1: Rectangle — red
    (0.25, 0.65, 0.85),   # Class 2: Circle — blue
    (0.25, 0.75, 0.35),   # Class 3: Triangle — green
]
CLASS_NAMES: list[str] = ["Background", "Rectangle", "Circle", "Triangle"]
SEG_CMAP = ListedColormap(PALETTE)

### 0.1 Synthetic Dataset Generation

We generate images containing 1–4 randomly placed coloured shapes on a near-black background. Each shape type corresponds to a unique class index in the segmentation mask:

| Class | Index | Colour |
|-------|-------|--------|
| Background | 0 | Dark grey |
| Rectangle | 1 | Red |
| Circle | 2 | Blue |
| Triangle | 3 | Green |

Each image is a float32 tensor of shape $(3, H, W)$ with pixel values in $[0, 1]$. Each mask is a long tensor of shape $(H, W)$ with integer class indices $\{0, 1, 2, 3\}$.

Shapes are drawn procedurally using NumPy, which gives us full control over the ground-truth mask. Overlapping shapes use **painter's algorithm** (later-drawn shapes overwrite earlier ones in the mask), so every pixel has exactly one class label.

In [ ]:
def draw_rectangle(
    image: np.ndarray,
    mask: np.ndarray,
    class_idx: int,
    colour: tuple[float, float, float],
    rng: np.random.Generator,
) -> None:
    """Draw a filled rectangle on image and update the mask in-place.

    Args:
        image: Float32 array of shape (H, W, 3) in [0, 1].
        mask: Int64 array of shape (H, W) with class indices.
        class_idx: Integer class label to write into the mask.
        colour: RGB tuple for the rectangle fill colour.
        rng: NumPy random generator for reproducible placement.
    """
    height, width = image.shape[:2]
    shape_h = rng.integers(MIN_SHAPE_SIZE, MAX_SHAPE_SIZE + 1)
    shape_w = rng.integers(MIN_SHAPE_SIZE, MAX_SHAPE_SIZE + 1)
    top = rng.integers(0, height - shape_h)
    left = rng.integers(0, width - shape_w)
    image[top : top + shape_h, left : left + shape_w] = colour
    mask[top : top + shape_h, left : left + shape_w] = class_idx


def draw_circle(
    image: np.ndarray,
    mask: np.ndarray,
    class_idx: int,
    colour: tuple[float, float, float],
    rng: np.random.Generator,
) -> None:
    """Draw a filled circle on image and update the mask in-place.

    Args:
        image: Float32 array of shape (H, W, 3) in [0, 1].
        mask: Int64 array of shape (H, W) with class indices.
        class_idx: Integer class label to write into the mask.
        colour: RGB tuple for the circle fill colour.
        rng: NumPy random generator for reproducible placement.
    """
    height, width = image.shape[:2]
    radius = rng.integers(MIN_SHAPE_SIZE // 2, MAX_SHAPE_SIZE // 2 + 1)
    centre_y = rng.integers(radius, height - radius)
    centre_x = rng.integers(radius, width - radius)
    # Build pixel grid and fill circle using distance from centre
    ys, xs = np.ogrid[:height, :width]
    dist_sq = (ys - centre_y) ** 2 + (xs - centre_x) ** 2
    circle_mask = dist_sq <= radius ** 2
    image[circle_mask] = colour
    mask[circle_mask] = class_idx


def draw_triangle(
    image: np.ndarray,
    mask: np.ndarray,
    class_idx: int,
    colour: tuple[float, float, float],
    rng: np.random.Generator,
) -> None:
    """Draw a filled upward-pointing triangle on image and update the mask in-place.

    The triangle is defined by a bounding box; pixels are filled using
    a row-by-row scan line approach for simplicity and speed.

    Args:
        image: Float32 array of shape (H, W, 3) in [0, 1].
        mask: Int64 array of shape (H, W) with class indices.
        class_idx: Integer class label to write into the mask.
        colour: RGB tuple for the triangle fill colour.
        rng: NumPy random generator for reproducible placement.
    """
    height, width = image.shape[:2]
    tri_h = rng.integers(MIN_SHAPE_SIZE, MAX_SHAPE_SIZE + 1)
    tri_w = rng.integers(MIN_SHAPE_SIZE, MAX_SHAPE_SIZE + 1)
    top = rng.integers(0, height - tri_h)
    left = rng.integers(0, width - tri_w)
    apex_x = left + tri_w // 2
    # Scan each row: at row r below apex the half-width grows linearly
    for row in range(tri_h):
        half_w = int(round((row + 1) / tri_h * (tri_w / 2)))
        col_start = max(0, apex_x - half_w)
        col_end = min(width, apex_x + half_w + 1)
        pixel_row = top + row
        if 0 <= pixel_row < height:
            image[pixel_row, col_start:col_end] = colour
            mask[pixel_row, col_start:col_end] = class_idx


def generate_segmentation_sample(
    image_size: int,
    num_classes: int,
    max_shapes: int,
    rng: np.random.Generator,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate a single synthetic segmentation sample with random shapes.

    Creates an RGB image containing 1–max_shapes randomly placed
    coloured shapes (rectangles, circles, triangles) on a dark background.
    Each shape type maps to a unique class index in the segmentation mask.

    Args:
        image_size: Spatial resolution H = W.
        num_classes: Total number of classes including background (class 0).
        max_shapes: Maximum number of shapes to place in one image.
        rng: NumPy random generator for reproducibility.

    Returns:
        Tuple of:
          - image_tensor: Float32 tensor of shape (3, H, W) in [0, 1].
          - mask_tensor: Long tensor of shape (H, W) with class indices 0..num_classes-1.
    """
    # Background colour: near-black with slight noise
    image = np.full((image_size, image_size, 3), 0.08, dtype=np.float32)
    image += rng.uniform(0.0, 0.04, size=image.shape).astype(np.float32)
    mask = np.zeros((image_size, image_size), dtype=np.int64)

    # Shape colours (one per foreground class)
    # Class 1 = Rectangle (red), 2 = Circle (blue), 3 = Triangle (green)
    shape_colours: list[tuple[float, float, float]] = [
        (0.85 + rng.uniform(-0.05, 0.05), 0.20 + rng.uniform(-0.05, 0.05), 0.20 + rng.uniform(-0.05, 0.05)),
        (0.20 + rng.uniform(-0.05, 0.05), 0.55 + rng.uniform(-0.05, 0.05), 0.85 + rng.uniform(-0.05, 0.05)),
        (0.20 + rng.uniform(-0.05, 0.05), 0.75 + rng.uniform(-0.05, 0.05), 0.30 + rng.uniform(-0.05, 0.05)),
    ]
    draw_fns = [draw_rectangle, draw_circle, draw_triangle]

    num_shapes = rng.integers(1, max_shapes + 1)
    for _ in range(num_shapes):
        # Pick a foreground class uniformly at random (1..num_classes-1)
        class_idx = int(rng.integers(1, num_classes))
        draw_fn = draw_fns[class_idx - 1]
        colour = shape_colours[class_idx - 1]
        draw_fn(image, mask, class_idx, colour, rng)

    # Clip image to valid range
    image = np.clip(image, 0.0, 1.0)

    # Convert to PyTorch tensors, channels-first
    image_tensor = torch.from_numpy(image.transpose(2, 0, 1))  # (3, H, W)
    mask_tensor = torch.from_numpy(mask)  # (H, W) long
    return image_tensor, mask_tensor

In [ ]:
class SegmentationDataset(Dataset):
    """Dataset of synthetic segmentation samples.

    Wraps a pre-generated list of (image, mask) pairs and exposes the
    standard PyTorch Dataset interface.

    Attributes:
        samples: List of (image_tensor, mask_tensor) tuples.
    """

    def __init__(self, samples: list[tuple[torch.Tensor, torch.Tensor]]) -> None:
        """Initialise the dataset with pre-generated samples.

        Args:
            samples: List of (image_tensor, mask_tensor) tuples where
                image_tensor has shape (3, H, W) and mask_tensor has shape (H, W).
        """
        self.samples = samples

    def __len__(self) -> int:
        """Return the number of samples in the dataset."""
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Return the (image, mask) pair at position idx.

        Args:
            idx: Sample index.

        Returns:
            Tuple of (image_tensor, mask_tensor).
        """
        return self.samples[idx]


def build_dataset(
    num_samples: int,
    image_size: int,
    num_classes: int,
    max_shapes: int,
    seed: int,
) -> SegmentationDataset:
    """Generate a synthetic SegmentationDataset with a fixed random seed.

    Args:
        num_samples: Number of samples to generate.
        image_size: Spatial resolution H = W.
        num_classes: Number of classes including background.
        max_shapes: Maximum shapes per image.
        seed: Random seed for reproducibility.

    Returns:
        A SegmentationDataset containing num_samples (image, mask) pairs.
    """
    rng = np.random.default_rng(seed)
    samples: list[tuple[torch.Tensor, torch.Tensor]] = [
        generate_segmentation_sample(image_size, num_classes, max_shapes, rng)
        for _ in range(num_samples)
    ]
    return SegmentationDataset(samples)


# Build train and validation datasets
print("Generating training data...")
train_dataset = build_dataset(NUM_TRAIN, IMAGE_SIZE, NUM_CLASSES, MAX_SHAPES, seed=SEED)
print("Generating validation data...")
val_dataset = build_dataset(NUM_VAL, IMAGE_SIZE, NUM_CLASSES, MAX_SHAPES, seed=SEED + 1)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

# Inspect one sample
sample_image, sample_mask = train_dataset[0]
print(f"Image tensor shape: {sample_image.shape}, dtype: {sample_image.dtype}")
print(f"Mask tensor shape:  {sample_mask.shape}, dtype: {sample_mask.dtype}")
print(f"Unique class indices in sample mask: {sample_mask.unique().tolist()}")

In [ ]:
# ── EDA: Class pixel distribution and sample grid ─────────────────────────────
def compute_class_pixel_counts(
    dataset: SegmentationDataset,
    num_classes: int,
) -> np.ndarray:
    """Count pixels per class across the entire dataset.

    Args:
        dataset: The segmentation dataset to analyse.
        num_classes: Number of classes.

    Returns:
        Int64 array of shape (num_classes,) with pixel counts per class.
    """
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, mask in dataset:
        for cls in range(num_classes):
            counts[cls] += int((mask == cls).sum().item())
    return counts


pixel_counts = compute_class_pixel_counts(train_dataset, NUM_CLASSES)
total_pixels = pixel_counts.sum()
pixel_fractions = pixel_counts / total_pixels

print("Class pixel distribution (training set):")
for cls_idx, (name, count, frac) in enumerate(
    zip(CLASS_NAMES, pixel_counts, pixel_fractions)
):
    print(f"  Class {cls_idx} ({name}): {count:,} pixels ({frac:.1%})")

# Plot class pixel distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

bar_colours = [PALETTE[i] for i in range(NUM_CLASSES)]
axes[0].bar(CLASS_NAMES, pixel_fractions, color=bar_colours, edgecolor="white", linewidth=0.8)
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Fraction of Total Pixels")
axes[0].set_title("Class Pixel Distribution (Training Set)")
axes[0].set_ylim(0, 1.0)
for bar_i, frac in enumerate(pixel_fractions):
    axes[0].text(bar_i, frac + 0.01, f"{frac:.1%}", ha="center", fontsize=10)

# Sample grid: 4 images with their masks
num_show = 4
axes[1].axis("off")
axes[1].set_title("Sample Grid (image | mask)")
plt.tight_layout()
plt.show()

# Separate sample grid figure
fig2, grid_axes = plt.subplots(2, num_show, figsize=(14, 5))
for col in range(num_show):
    img, msk = train_dataset[col]
    # Channels-first to HWC for matplotlib
    img_np = img.permute(1, 2, 0).numpy()
    msk_np = msk.numpy()
    grid_axes[0, col].imshow(img_np)
    grid_axes[0, col].set_title(f"Image {col}")
    grid_axes[0, col].axis("off")
    grid_axes[1, col].imshow(msk_np, cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
    grid_axes[1, col].set_title(f"Mask {col}")
    grid_axes[1, col].axis("off")

# Legend for mask colours
legend_patches = [
    mpatches.Patch(facecolor=PALETTE[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)
]
fig2.legend(handles=legend_patches, loc="lower center", ncol=NUM_CLASSES, fontsize=10)
fig2.suptitle("Synthetic Segmentation Samples (top: RGB image, bottom: segmentation mask)", fontsize=12)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

# Verify batch shapes
batch_images, batch_masks = next(iter(train_loader))
print(f"Batch images: {batch_images.shape}  (B, C, H, W)")
print(f"Batch masks:  {batch_masks.shape}   (B, H, W)")

---
## Part 1 — Metrics & Upsampling Techniques

Before building the networks, we need two things:

1. **Evaluation metrics** that measure segmentation quality beyond pixel accuracy.
2. **Upsampling methods** — the core building block that allows a segmentation network to recover spatial resolution after downsampling in the encoder.

### 1.1 Segmentation Metrics

**Pixel accuracy** is the fraction of pixels classified correctly:

$$\text{PixelAcc} = \frac{\sum_c |\{p : \hat{y}_p = c, y_p = c\}|}{HW}$$

However, pixel accuracy is misleading when classes are imbalanced (e.g., if background covers 80% of pixels, a model that predicts only background achieves 80% accuracy). **Intersection over Union (IoU)** per class is more informative:

$$\text{IoU}_c = \frac{|\hat{Y}_c \cap Y_c|}{|\hat{Y}_c \cup Y_c|} = \frac{TP_c}{TP_c + FP_c + FN_c}$$

**Mean IoU (mIoU)** averages over classes that actually appear in the ground truth:

$$\text{mIoU} = \frac{1}{|\mathcal{C}_{present}|} \sum_{c \in \mathcal{C}_{present}} \text{IoU}_c$$

In [ ]:
def compute_iou_per_class(
    pred_mask: torch.Tensor,
    gt_mask: torch.Tensor,
    num_classes: int,
) -> torch.Tensor:
    """Compute per-class Intersection over Union.

    For each class c, IoU_c = TP / (TP + FP + FN) where:
      TP = pixels predicted as c and labelled c
      FP = pixels predicted as c but labelled differently
      FN = pixels labelled c but predicted differently

    Args:
        pred_mask: Predicted class indices, shape (H, W) or (B, H, W), long.
        gt_mask: Ground-truth class indices, same shape as pred_mask, long.
        num_classes: Total number of classes C.

    Returns:
        Float tensor of shape (C,) with IoU for each class.
        Classes absent from gt_mask have IoU = NaN.
    """
    iou_scores = torch.full((num_classes,), float("nan"))
    pred_flat = pred_mask.reshape(-1)
    gt_flat = gt_mask.reshape(-1)

    for cls in range(num_classes):
        pred_cls = pred_flat == cls
        gt_cls = gt_flat == cls

        # Skip classes not present in ground truth
        if gt_cls.sum() == 0:
            continue

        intersection = (pred_cls & gt_cls).sum().float()
        union = (pred_cls | gt_cls).sum().float()
        iou_scores[cls] = intersection / (union + 1e-8)

    return iou_scores


def compute_mean_iou(
    pred_mask: torch.Tensor,
    gt_mask: torch.Tensor,
    num_classes: int,
) -> float:
    """Compute mean IoU averaged over classes present in the ground truth.

    Args:
        pred_mask: Predicted class indices, shape (H, W) or (B, H, W), long.
        gt_mask: Ground-truth class indices, same shape as pred_mask, long.
        num_classes: Total number of classes C.

    Returns:
        Scalar float: mean IoU over present classes. Returns 0.0 if no
        classes are present.
    """
    iou_per_class = compute_iou_per_class(pred_mask, gt_mask, num_classes)
    valid = ~torch.isnan(iou_per_class)
    if valid.sum() == 0:
        return 0.0
    return iou_per_class[valid].mean().item()


def pixel_accuracy(
    pred_mask: torch.Tensor,
    gt_mask: torch.Tensor,
) -> float:
    """Compute fraction of correctly classified pixels.

    Args:
        pred_mask: Predicted class indices, shape (H, W) or (B, H, W), long.
        gt_mask: Ground-truth class indices, same shape, long.

    Returns:
        Scalar float in [0, 1].
    """
    correct = (pred_mask == gt_mask).sum().item()
    total = gt_mask.numel()
    return correct / total

In [ ]:
# ── Metric verification on trivial cases ─────────────────────────────────────
# Perfect prediction → IoU = 1.0 for all present classes
perfect_gt = torch.tensor([[0, 1, 2, 3], [0, 0, 1, 1]], dtype=torch.long)
perfect_pred = perfect_gt.clone()
iou_perfect = compute_iou_per_class(perfect_pred, perfect_gt, NUM_CLASSES)
print("Perfect prediction IoU per class:", [f"{v:.3f}" for v in iou_perfect.tolist()])
print(f"Mean IoU (perfect): {compute_mean_iou(perfect_pred, perfect_gt, NUM_CLASSES):.4f}")
print(f"Pixel accuracy (perfect): {pixel_accuracy(perfect_pred, perfect_gt):.4f}")

# Worst case: predict background everywhere → foreground IoU = 0
worst_pred = torch.zeros_like(perfect_gt)
iou_worst = compute_iou_per_class(worst_pred, perfect_gt, NUM_CLASSES)
print("\nAll-background prediction IoU per class:", [f"{v:.3f}" if not math.isnan(v) else "NaN" for v in iou_worst.tolist()])
print(f"Mean IoU (all-background): {compute_mean_iou(worst_pred, perfect_gt, NUM_CLASSES):.4f}")
print(f"Pixel accuracy (all-background): {pixel_accuracy(worst_pred, perfect_gt):.4f}")

### 1.2 Upsampling: Transposed Convolution vs Bilinear Interpolation

Segmentation networks downsample the input (via pooling or strided convolutions) to extract high-level features, then must **upsample** the feature maps back to the input resolution to produce per-pixel predictions. Two common approaches exist:

**Transposed Convolution** (`nn.ConvTranspose2d`)
- Also called *deconvolution* (informally) or *fractionally-strided convolution*.
- Learnable parameters: the network can learn what information to distribute during upsampling.
- **Checkerboard artifact:** When `kernel_size` is not divisible by `stride`, output pixels are unevenly covered by the kernel, producing a grid pattern in the output — visually apparent in generated images and early FCN predictions.
- Formula: output size $= (input\_size - 1) \times stride - 2 \times padding + kernel\_size$.

**Bilinear Interpolation** (`nn.Upsample(mode='bilinear')`)
- Fixed: no learnable parameters — uses weighted average of 4 neighbouring pixels.
- Smooth output: no checkerboard artifacts.
- Typically followed by a regular `Conv2d` to refine features (the *upsample-then-convolve* pattern).
- U-Net's decoder commonly uses bilinear upsampling + `DoubleConv` for stability and quality.

In [ ]:
# ── Upsampling comparison: transposed conv vs bilinear ────────────────────────
torch.manual_seed(SEED)

# Simulate a low-resolution feature map (1 channel for easy visualisation)
low_res = torch.randn(1, 1, 8, 8)  # (B=1, C=1, H=8, W=8)

# Option A: Transposed convolution 2× upsampling (kernel=4, stride=2, pad=1)
# kernel_size=4 is NOT divisible by stride=2 → checkerboard artifacts expected
transposed_conv = nn.ConvTranspose2d(
    in_channels=1, out_channels=1, kernel_size=4, stride=2, padding=1, bias=False
)
nn.init.ones_(transposed_conv.weight)  # Uniform weights to exaggerate artifacts
out_transposed = transposed_conv(low_res)

# Option B: Bilinear interpolation 2×
bilinear_up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
out_bilinear = bilinear_up(low_res)

print(f"Input shape:              {low_res.shape}")
print(f"Transposed conv output:   {out_transposed.shape}")
print(f"Bilinear output:          {out_bilinear.shape}")

# Visualise the two approaches on the same feature map
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Helper to detach and squeeze for display
def to_display(t: torch.Tensor) -> np.ndarray:
    """Convert a (1, 1, H, W) tensor to a (H, W) numpy array for display."""
    return t.detach().squeeze().numpy()

axes[0].imshow(to_display(low_res), cmap="viridis")
axes[0].set_title("Input Feature Map (8×8)")
axes[0].axis("off")

im1 = axes[1].imshow(to_display(out_transposed), cmap="viridis")
axes[1].set_title("Transposed Conv (16×16)\n(checkerboard pattern)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(to_display(out_bilinear), cmap="viridis")
axes[2].set_title("Bilinear Upsample (16×16)\n(smooth)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle("Transposed Convolution vs Bilinear Upsampling", fontsize=13)
plt.tight_layout()
plt.show()

# Quantify the non-uniformity
trans_std = to_display(out_transposed).std()
bil_std = to_display(out_bilinear).std()
print(f"\nOutput std (transposed conv):   {trans_std:.4f}  (higher → more artifact pattern)")
print(f"Output std (bilinear):          {bil_std:.4f}  (lower  → smoother)")

---
## Part 2 — Fully Convolutional Networks (FCN)

The FCN family, introduced by Long et al. (2015), converts a standard classification CNN into a segmentation network by:
1. **Replacing fully-connected layers with 1×1 convolutions** — making the network fully convolutional so it accepts arbitrary input sizes.
2. **Upsampling the coarse score map** back to input resolution using transposed convolutions.
3. **Adding skip connections** from intermediate feature maps to recover fine spatial detail lost during downsampling.

### 2.1 FCN Architecture Variants

Our encoder downsamples $64 \times 64$ input to $8 \times 8$ (three $2\times$ downsampling steps = $8\times$ total):

| Variant | Upsampling Strategy | Spatial Detail |
|---------|--------------------|-----------------|
| **FCN-32s** | Single $32\times$ transposed conv from $8\times 8$ to $64\times 64$ | Coarse |
| **FCN-8s** | Progressive $2\times$ upsamplings with skip connections from stride-16 and stride-8 maps | Fine |

The skip connections in FCN-8s fuse **deep, semantic** features with **shallow, spatial** features — recovering detail that the coarser FCN-32s loses.

$$\text{score}_{8s} = \text{upsample}_{2\times}(\text{upsample}_{2\times}(\text{score}_{32}) + \text{skip}_{16}) + \text{skip}_8$$

In [ ]:
class FCNEncoder(nn.Module):
    """Shared encoder for FCN variants.

    Three convolutional blocks, each followed by MaxPool2d(stride=2),
    producing feature maps at three spatial scales:
      - After block 1 + pool: stride 2  (32×32 for 64×64 input)
      - After block 2 + pool: stride 4  (16×16 for 64×64 input)
      - After block 3 + pool: stride 8  (8×8  for 64×64 input)

    Channels: 3 → 32 → 64 → 128.

    Attributes:
        block1: First conv block (3 → 32 channels).
        pool1:  Max pooling after block 1.
        block2: Second conv block (32 → 64 channels).
        pool2:  Max pooling after block 2.
        block3: Third conv block (64 → 128 channels).
        pool3:  Max pooling after block 3.
    """

    def __init__(self) -> None:
        """Initialise FCNEncoder with three downsampling blocks."""
        super().__init__()
        # Block 1: 3 → 32 channels, spatial: 64 → 32
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: 32 → 64 channels, spatial: 32 → 16
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 3: 64 → 128 channels, spatial: 16 → 8
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Run the encoder and return feature maps at three scales.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Tuple (feat_stride4, feat_stride8_pre_pool, feat_stride8) where:
              - feat_stride4: Features after pool1 (B, 32, H/2, W/2).
              - feat_stride8_pre_pool: Features after block2, before pool2 used as
                skip at stride-4 (B, 64, H/4, W/4).
              - feat_stride32: Deepest features after pool3 (B, 128, H/8, W/8).
        """
        feat1 = self.block1(x)        # (B, 32, H, W)
        feat1_p = self.pool1(feat1)   # (B, 32, H/2, W/2)  — stride-2 map

        feat2 = self.block2(feat1_p)  # (B, 64, H/2, W/2)
        feat2_p = self.pool2(feat2)   # (B, 64, H/4, W/4)  — stride-4 map

        feat3 = self.block3(feat2_p)  # (B, 128, H/4, W/4)
        feat3_p = self.pool3(feat3)   # (B, 128, H/8, W/8) — stride-8 map

        return feat2_p, feat3, feat3_p

### 2.2 FCN-32s: Coarse Single-Shot Upsampling

FCN-32s takes the deepest feature map (stride-8 in our case, i.e., $8 \times 8$ for $64 \times 64$ input) and directly upsamples it back to input resolution with a single large transposed convolution. The upsampling factor equals the total stride of the encoder.

For our encoder with total stride $8$:

$$\text{kernel\_size} = \text{stride} \times 2, \quad \text{padding} = \text{stride} \div 2$$

Output shape: $(B, C_{\text{classes}}, H, W)$ — one score per pixel per class.

In [ ]:
class FCN32s(nn.Module):
    """Fully Convolutional Network — 32s (single large upsampling) variant.

    Takes the deepest encoder feature map and applies a single transposed
    convolution to recover the input spatial resolution in one shot.
    This produces coarse predictions that lack fine boundary detail.

    Attributes:
        encoder: Shared FCNEncoder backbone.
        score_conv: 1×1 convolution projecting 128 channels → num_classes.
        upsample: Transposed conv upsampling from stride-8 to full resolution.
    """

    def __init__(self, num_classes: int, encoder_stride: int = 8) -> None:
        """Initialise FCN-32s.

        Args:
            num_classes: Number of output classes (including background).
            encoder_stride: Total spatial downsampling factor of the encoder.
        """
        super().__init__()
        self.encoder = FCNEncoder()
        # 1×1 conv: project 128 deep features to class scores
        self.score_conv = nn.Conv2d(128, num_classes, kernel_size=1)
        # Single transposed conv: stride-8 map → full resolution
        # kernel = 2 * stride, padding = stride // 2 for clean upsampling
        self.upsample = nn.ConvTranspose2d(
            num_classes,
            num_classes,
            kernel_size=encoder_stride * 2,
            stride=encoder_stride,
            padding=encoder_stride // 2,
            bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: image → per-pixel class logits.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Logit tensor of shape (B, num_classes, H, W).
        """
        _, _, feat_deep = self.encoder(x)   # (B, 128, H/8, W/8)
        scores = self.score_conv(feat_deep) # (B, C, H/8, W/8)
        logits = self.upsample(scores)      # (B, C, H, W)
        return logits


# Sanity check: output shape matches input
fcn32s_check = FCN32s(NUM_CLASSES).to(device)
dummy_input = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
dummy_out = fcn32s_check(dummy_input)
print(f"FCN-32s input:  {dummy_input.shape}")
print(f"FCN-32s output: {dummy_out.shape}   (expected: (2, {NUM_CLASSES}, {IMAGE_SIZE}, {IMAGE_SIZE}))")
assert dummy_out.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Shape mismatch!"
num_params_fcn32s = sum(p.numel() for p in fcn32s_check.parameters() if p.requires_grad)
print(f"FCN-32s trainable parameters: {num_params_fcn32s:,}")

### 2.3 FCN-8s: Progressive Upsampling with Skip Connections

FCN-8s improves upon FCN-32s by **fusing predictions from multiple resolution levels** via skip connections. Rather than upsampling everything in one shot, it:

1. Produces a coarse score from the deepest features (stride-8 map, $8 \times 8$).
2. Upsamples $2\times$ to $16 \times 16$ and **adds** a score projection from the stride-4 features.
3. Upsamples $2\times$ to $32 \times 32$ and **adds** a score projection from the stride-2 features (if available).
4. Upsamples $2\times$ to $64 \times 64$ for final output.

Each skip connection contributes features from an earlier, less-downsampled layer — these shallow features retain fine spatial structure (edges, textures) that deeper layers have abstracted away. The result is sharper segmentation boundaries.

In [ ]:
class FCN8s(nn.Module):
    """Fully Convolutional Network — 8s (progressive skip-connection) variant.

    Adds skip connections from intermediate encoder feature maps to recover
    finer spatial detail compared to FCN-32s. Uses three progressive 2×
    transposed convolution upsamplings, each fused with a skip score.

    Architecture:
      1. score_deep:  1×1 conv  128 → C  on stride-8 (deepest) features
      2. up2:         ConvTranspose2d 2× → stride-4 scale
      3. score_skip4: 1×1 conv   64 → C  on stride-4 features (skip)
      4. up4:         ConvTranspose2d 2× → stride-2 scale
      5. score_skip2: 1×1 conv   64 → C  on stride-2 features (skip)
      6. up8:         ConvTranspose2d 2× → full resolution

    Attributes:
        encoder: Shared FCNEncoder backbone.
        score_deep: 1×1 conv projecting deep features to class scores.
        up2: 2× transposed convolution for first upsampling step.
        score_skip4: 1×1 conv projecting stride-4 skip features.
        up4: 2× transposed convolution for second upsampling step.
        score_skip2: 1×1 conv projecting stride-2 skip features.
        up8: 2× transposed convolution for final upsampling step.
    """

    def __init__(self, num_classes: int) -> None:
        """Initialise FCN-8s with encoder and three skip-connection upsamplings.

        Args:
            num_classes: Number of output classes (including background).
        """
        super().__init__()
        self.encoder = FCNEncoder()

        # Score projection for deepest features (stride-8, 128 channels)
        self.score_deep = nn.Conv2d(128, num_classes, kernel_size=1)

        # First 2× upsample: stride-8 → stride-4 scale
        self.up2 = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False
        )
        # Skip score from stride-4 features (feat2_p: 64 channels)
        self.score_skip4 = nn.Conv2d(64, num_classes, kernel_size=1)

        # Second 2× upsample: stride-4 → stride-2 scale
        self.up4 = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False
        )
        # Skip score from stride-2 features (feat3: 128 channels before pool3)
        # Note: feat3 is at stride-4 spatial size from block3 before pool3
        self.score_skip2 = nn.Conv2d(128, num_classes, kernel_size=1)

        # Final 2× upsample: stride-2 → full resolution
        self.up8 = nn.ConvTranspose2d(
            num_classes, num_classes, kernel_size=4, stride=2, padding=1, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with progressive skip-connection fusion.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Logit tensor of shape (B, num_classes, H, W).
        """
        # Encoder: three feature maps
        # feat_stride4: (B, 64,  H/4, W/4)  — after block2, before pool2
        # feat_stride4_deep: (B, 128, H/4, W/4)  — after block3, before pool3
        # feat_deep:    (B, 128, H/8, W/8)  — after pool3
        feat_stride4, feat_stride4_deep, feat_deep = self.encoder(x)

        # Step 1: Score deep features at stride-8
        score = self.score_deep(feat_deep)          # (B, C, H/8, W/8)

        # Step 2: Upsample 2× to stride-4, fuse with stride-4 skip
        score = self.up2(score)                     # (B, C, H/4, W/4)
        skip4 = self.score_skip4(feat_stride4)      # (B, C, H/4, W/4)
        score = score + skip4                       # Element-wise addition

        # Step 3: Upsample 2× to stride-2, fuse with stride-2 skip
        score = self.up4(score)                     # (B, C, H/2, W/2)
        skip2 = self.score_skip2(feat_stride4_deep) # (B, C, H/4, W/4)
        # Upsample skip2 to match score spatial size
        skip2_up = F.interpolate(
            skip2, size=score.shape[2:], mode="bilinear", align_corners=False
        )
        score = score + skip2_up                    # Fuse at stride-2 scale

        # Step 4: Final 2× upsample to full resolution
        logits = self.up8(score)                    # (B, C, H, W)
        return logits


# Sanity check
fcn8s_check = FCN8s(NUM_CLASSES).to(device)
dummy_out_8s = fcn8s_check(dummy_input)
print(f"FCN-8s input:  {dummy_input.shape}")
print(f"FCN-8s output: {dummy_out_8s.shape}   (expected: (2, {NUM_CLASSES}, {IMAGE_SIZE}, {IMAGE_SIZE}))")
assert dummy_out_8s.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Shape mismatch!"
num_params_fcn8s = sum(p.numel() for p in fcn8s_check.parameters() if p.requires_grad)
print(f"FCN-8s trainable parameters: {num_params_fcn8s:,}")

---
## Part 3 — U-Net: Symmetric Encoder-Decoder with Skip Connections

U-Net (Ronneberger et al., 2015) takes the skip-connection idea to its logical conclusion: it builds a **fully symmetric encoder-decoder** where every encoder level is mirrored by a corresponding decoder level, connected by a skip connection that **concatenates** (not just adds) features.

The key architectural differences from FCN:

| Feature | FCN-8s | U-Net |
|---------|--------|-------|
| Skip connections | Addition at 2 scales | Concatenation at every scale |
| Decoder | Single linear upsample path | Symmetric 4-level decoder |
| Feature reuse | Score projections | Full feature concatenation |
| Upsampling | Transposed conv | Bilinear + DoubleConv |

By concatenating full feature maps (not just scalar-projected scores), U-Net's decoder has access to **complete spatial information** at every scale — making it particularly effective for small-object segmentation and precise boundary delineation.

### 3.1 Building Block: DoubleConv

U-Net's basic unit is two consecutive `Conv2d + BatchNorm + ReLU` blocks, applied at every encoder and decoder level. This doubles the receptive field per level while keeping spatial size constant.

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive (Conv2d → BatchNorm → ReLU) blocks.

    This is the basic building block for both encoder and decoder stages
    of U-Net. It preserves spatial dimensions (padding=1) while transforming
    the channel count from in_channels to out_channels.

    Attributes:
        conv_block: Sequential of two Conv-BN-ReLU triples.
    """

    def __init__(self, in_channels: int, out_channels: int) -> None:
        """Initialise DoubleConv block.

        Args:
            in_channels: Number of input feature map channels.
            out_channels: Number of output feature map channels.
        """
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply double conv block.

        Args:
            x: Input tensor of shape (B, in_channels, H, W).

        Returns:
            Output tensor of shape (B, out_channels, H, W).
        """
        return self.conv_block(x)


class UNetEncoder(nn.Module):
    """U-Net encoder: four downsampling stages with skip feature storage.

    Each stage applies DoubleConv then MaxPool2d(2) for 2× downsampling.
    The pre-pool feature map at each level is saved as a skip connection
    for the corresponding decoder stage.

    Channel progression: 3 → 32 → 64 → 128 → 256.
    Spatial progression (for 64×64 input): 64 → 32 → 16 → 8 → 4.

    Attributes:
        enc1: First DoubleConv block (3 → 32 channels).
        enc2: Second DoubleConv block (32 → 64 channels).
        enc3: Third DoubleConv block (64 → 128 channels).
        enc4: Fourth DoubleConv block (128 → 256 channels).
        pool: MaxPool2d(2) shared across all stages.
    """

    def __init__(self) -> None:
        """Initialise UNetEncoder with four downsampling DoubleConv blocks."""
        super().__init__()
        self.enc1 = DoubleConv(3, 32)      # 64×64 → 64×64 (pre-pool)
        self.enc2 = DoubleConv(32, 64)     # 32×32 → 32×32 (pre-pool)
        self.enc3 = DoubleConv(64, 128)    # 16×16 → 16×16 (pre-pool)
        self.enc4 = DoubleConv(128, 256)   # 8×8   → 8×8   (pre-pool)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Run encoder and return bottleneck features plus four skip connections.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Tuple (skip1, skip2, skip3, skip4, bottleneck) where:
              skip1: (B, 32,  H,    W)    — after enc1, before pool
              skip2: (B, 64,  H/2,  W/2)  — after enc2, before pool
              skip3: (B, 128, H/4,  W/4)  — after enc3, before pool
              skip4: (B, 256, H/8,  W/8)  — after enc4, before pool
              bottleneck: (B, 256, H/16, W/16) — deepest feature map after pool
        """
        skip1 = self.enc1(x)              # (B, 32,  H,    W)
        x = self.pool(skip1)              # (B, 32,  H/2,  W/2)

        skip2 = self.enc2(x)              # (B, 64,  H/2,  W/2)
        x = self.pool(skip2)              # (B, 64,  H/4,  W/4)

        skip3 = self.enc3(x)              # (B, 128, H/4,  W/4)
        x = self.pool(skip3)              # (B, 128, H/8,  W/8)

        skip4 = self.enc4(x)              # (B, 256, H/8,  W/8)
        bottleneck = self.pool(skip4)     # (B, 256, H/16, W/16)

        return skip1, skip2, skip3, skip4, bottleneck


class UNetDecoder(nn.Module):
    """U-Net decoder: four upsampling stages, each concatenating a skip connection.

    Each stage bilinearly upsamples 2×, concatenates the corresponding
    encoder skip features, then applies a DoubleConv to refine.

    Channel progression (after concatenation and double-conv):
      512 → 256 → 128 → 64 → 32.

    Attributes:
        up4: Upsample + DoubleConv for level 4 (512 → 256 channels).
        up3: Upsample + DoubleConv for level 3 (256 → 128 channels).
        up2: Upsample + DoubleConv for level 2 (128 → 64 channels).
        up1: Upsample + DoubleConv for level 1 (64 → 32 channels).
    """

    def __init__(self) -> None:
        """Initialise UNetDecoder with four upsampling DoubleConv blocks."""
        super().__init__()
        # Each decoder stage: concat(upsample, skip) → double_conv
        # Input channels = upsampled_channels + skip_channels
        self.up4 = DoubleConv(256 + 256, 256)   # bottleneck(256) + skip4(256) → 256
        self.up3 = DoubleConv(256 + 128, 128)   # dec4(256) + skip3(128) → 128
        self.up2 = DoubleConv(128 + 64, 64)     # dec3(128) + skip2(64) → 64
        self.up1 = DoubleConv(64 + 32, 32)      # dec2(64)  + skip1(32) → 32
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(
        self,
        skip1: torch.Tensor,
        skip2: torch.Tensor,
        skip3: torch.Tensor,
        skip4: torch.Tensor,
        bottleneck: torch.Tensor,
    ) -> torch.Tensor:
        """Run decoder with skip-connection fusion via concatenation.

        Args:
            skip1: Encoder level-1 features (B, 32,  H,    W).
            skip2: Encoder level-2 features (B, 64,  H/2,  W/2).
            skip3: Encoder level-3 features (B, 128, H/4,  W/4).
            skip4: Encoder level-4 features (B, 256, H/8,  W/8).
            bottleneck: Deepest features    (B, 256, H/16, W/16).

        Returns:
            Decoded feature tensor of shape (B, 32, H, W).
        """
        # Level 4: upsample bottleneck, concat skip4
        x = self.upsample(bottleneck)             # (B, 256, H/8, W/8)
        x = torch.cat([x, skip4], dim=1)          # (B, 512, H/8, W/8)
        x = self.up4(x)                           # (B, 256, H/8, W/8)

        # Level 3: upsample, concat skip3
        x = self.upsample(x)                      # (B, 256, H/4, W/4)
        x = torch.cat([x, skip3], dim=1)          # (B, 384, H/4, W/4)
        x = self.up3(x)                           # (B, 128, H/4, W/4)

        # Level 2: upsample, concat skip2
        x = self.upsample(x)                      # (B, 128, H/2, W/2)
        x = torch.cat([x, skip2], dim=1)          # (B, 192, H/2, W/2)
        x = self.up2(x)                           # (B, 64,  H/2, W/2)

        # Level 1: upsample, concat skip1
        x = self.upsample(x)                      # (B, 64,  H, W)
        x = torch.cat([x, skip1], dim=1)          # (B, 96,  H, W)
        x = self.up1(x)                           # (B, 32,  H, W)

        return x


class UNet(nn.Module):
    """Complete U-Net encoder-decoder with symmetric skip connections.

    Combines UNetEncoder and UNetDecoder with a final 1×1 convolution
    to produce per-pixel class logits.

    Attributes:
        encoder: UNetEncoder for feature extraction and skip connections.
        decoder: UNetDecoder for progressive upsampling with skip fusion.
        head: Final 1×1 conv projecting 32 channels to num_classes.
    """

    def __init__(self, num_classes: int) -> None:
        """Initialise U-Net.

        Args:
            num_classes: Number of output segmentation classes.
        """
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: image → per-pixel class logits.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Logit tensor of shape (B, num_classes, H, W).
        """
        skip1, skip2, skip3, skip4, bottleneck = self.encoder(x)
        decoded = self.decoder(skip1, skip2, skip3, skip4, bottleneck)
        logits = self.head(decoded)   # (B, num_classes, H, W)
        return logits


# Sanity check
unet_check = UNet(NUM_CLASSES).to(device)
unet_out = unet_check(dummy_input)
print(f"U-Net input:  {dummy_input.shape}")
print(f"U-Net output: {unet_out.shape}   (expected: (2, {NUM_CLASSES}, {IMAGE_SIZE}, {IMAGE_SIZE}))")
assert unet_out.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Shape mismatch!"
num_params_unet = sum(p.numel() for p in unet_check.parameters() if p.requires_grad)
print(f"U-Net trainable parameters: {num_params_unet:,}")

In [ ]:
# ── Parameter count comparison ────────────────────────────────────────────────
model_params = {
    "FCN-32s": num_params_fcn32s,
    "FCN-8s": num_params_fcn8s,
    "U-Net": num_params_unet,
}
print("Model parameter counts:")
print(f"  {'Model':<12} {'Parameters':>12}")
print(f"  {'------':<12} {'----------':>12}")
for model_name, param_count in model_params.items():
    print(f"  {model_name:<12} {param_count:>12,}")

### 3.2 Visualising U-Net Skip Connections

To build intuition for what skip connections contribute, we can visualise the encoder feature maps at each resolution level. Shallow features (level 1) retain edges and colour boundaries; deep features (level 4/bottleneck) capture semantic shape information at low resolution.

The decoder's concatenation operation merges these two types of information at every level — this is why U-Net produces sharper boundaries than FCN-32s, which only has access to deep features.

In [ ]:
# ── Visualise encoder skip features for one sample ───────────────────────────
unet_check.eval()
sample_img, sample_msk = val_dataset[0]
sample_img_batch = sample_img.unsqueeze(0).to(device)  # (1, 3, H, W)

with torch.no_grad():
    skip1_feat, skip2_feat, skip3_feat, skip4_feat, bottleneck_feat = (
        unet_check.encoder(sample_img_batch)
    )

skip_features = {
    f"Skip 1 (32ch, {skip1_feat.shape[2]}×{skip1_feat.shape[3]})": skip1_feat,
    f"Skip 2 (64ch, {skip2_feat.shape[2]}×{skip2_feat.shape[3]})": skip2_feat,
    f"Skip 3 (128ch, {skip3_feat.shape[2]}×{skip3_feat.shape[3]})": skip3_feat,
    f"Skip 4 (256ch, {skip4_feat.shape[2]}×{skip4_feat.shape[3]})": skip4_feat,
    f"Bottleneck (256ch, {bottleneck_feat.shape[2]}×{bottleneck_feat.shape[3]})": bottleneck_feat,
}

fig, axes = plt.subplots(2, 5, figsize=(16, 6))

# Top row: input image and its mask
axes[0, 0].imshow(sample_img.permute(1, 2, 0).numpy())
axes[0, 0].set_title("Input Image (64×64)")
axes[0, 0].axis("off")
axes[1, 0].imshow(sample_msk.numpy(), cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[1, 0].set_title("GT Mask")
axes[1, 0].axis("off")

# Bottom row: mean activation of each skip level
for col_i, (feat_name, feat_tensor) in enumerate(skip_features.items(), start=0):
    # Average across channels to get a 2D activation map
    act_map = feat_tensor[0].mean(dim=0).cpu().numpy()
    row_idx = 0 if col_i < 4 else 1
    ax_col = col_i + 1 if col_i < 4 else 0
    if col_i < 4:
        axes[0, col_i + 1].imshow(act_map, cmap="viridis")
        axes[0, col_i + 1].set_title(feat_name, fontsize=8)
        axes[0, col_i + 1].axis("off")
    else:
        axes[1, 1].imshow(act_map, cmap="viridis")
        axes[1, 1].set_title(feat_name, fontsize=8)
        axes[1, 1].axis("off")
        # Hide remaining axes
        for unused_col in range(2, 5):
            axes[1, unused_col].axis("off")

fig.suptitle("U-Net Encoder Skip Features (mean channel activation)", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 4 — Training All Three Models

We now train FCN-32s, FCN-8s, and U-Net on the synthetic segmentation dataset.

**Loss function:** `nn.CrossEntropyLoss()` with no class weighting. This loss accepts:
- **Logits:** $(B, C, H, W)$ — one score per pixel per class
- **Targets:** $(B, H, W)$ — integer class index per pixel

It internally applies softmax and then computes per-pixel negative log-likelihood, averaged over all pixels in the batch. This is exactly the pixel-wise cross-entropy loss used in the original FCN paper.

**Training conventions** (following RULES_CORE.md Section 10):
- Optimizer: `Adam` with `lr=1e-3`
- Metrics tracked per epoch: loss, pixel accuracy, mean IoU
- Best model selected by lowest validation loss

### Part 3 — U-Net Architecture

U-Net extends the FCN idea with a symmetric encoder-decoder structure and skip connections that concatenate high-resolution encoder features with upsampled decoder features, preserving fine spatial detail for precise segmentation.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float]:
    """Train the segmentation model for one epoch.

    Computes pixel-wise cross-entropy loss, pixel accuracy, and mean IoU
    over all batches. Accumulates per-sample metrics and returns averages.

    Args:
        model: Segmentation model (FCN or U-Net) producing (B, C, H, W) logits.
        dataloader: Training DataLoader yielding (images, masks) batches.
        optimizer: Optimizer instance (Adam).
        criterion: Loss function (nn.CrossEntropyLoss).
        device: Device to run on.

    Returns:
        Tuple of (avg_loss, pixel_acc, mean_iou) for the epoch.
    """
    model.train()
    total_loss = 0.0
    total_correct_pixels = 0
    total_pixels = 0
    total_iou = 0.0
    num_batches = 0

    for batch_images, batch_masks in dataloader:
        batch_images = batch_images.to(device)
        batch_masks = batch_masks.to(device)  # (B, H, W) long

        optimizer.zero_grad()
        logits = model(batch_images)           # (B, C, H, W)
        loss = criterion(logits, batch_masks)  # Pixel-wise cross-entropy
        loss.backward()
        optimizer.step()

        # Accumulate loss (weighted by batch size)
        batch_size = batch_images.size(0)
        total_loss += loss.item() * batch_size

        # Pixel accuracy
        pred_masks = logits.argmax(dim=1)      # (B, H, W)
        total_correct_pixels += (pred_masks == batch_masks).sum().item()
        total_pixels += batch_masks.numel()

        # Mean IoU (batch-level, averaged later)
        batch_miou = compute_mean_iou(
            pred_masks.cpu(), batch_masks.cpu(), NUM_CLASSES
        )
        total_iou += batch_miou
        num_batches += 1

    avg_loss = total_loss / len(dataloader.dataset)
    avg_pixel_acc = total_correct_pixels / total_pixels
    avg_mean_iou = total_iou / num_batches
    return avg_loss, avg_pixel_acc, avg_mean_iou


def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float]:
    """Evaluate the segmentation model on a validation or test set.

    Args:
        model: Segmentation model producing (B, C, H, W) logits.
        dataloader: Validation or test DataLoader.
        criterion: Loss function (nn.CrossEntropyLoss).
        device: Device to run on.

    Returns:
        Tuple of (avg_loss, pixel_acc, mean_iou) over the full dataset.
    """
    model.eval()
    total_loss = 0.0
    total_correct_pixels = 0
    total_pixels = 0
    total_iou = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch_images, batch_masks in dataloader:
            batch_images = batch_images.to(device)
            batch_masks = batch_masks.to(device)

            logits = model(batch_images)
            loss = criterion(logits, batch_masks)

            batch_size = batch_images.size(0)
            total_loss += loss.item() * batch_size

            pred_masks = logits.argmax(dim=1)
            total_correct_pixels += (pred_masks == batch_masks).sum().item()
            total_pixels += batch_masks.numel()

            batch_miou = compute_mean_iou(
                pred_masks.cpu(), batch_masks.cpu(), NUM_CLASSES
            )
            total_iou += batch_miou
            num_batches += 1

    avg_loss = total_loss / len(dataloader.dataset)
    avg_pixel_acc = total_correct_pixels / total_pixels
    avg_mean_iou = total_iou / num_batches
    return avg_loss, avg_pixel_acc, avg_mean_iou

In [ ]:
# ── Training: FCN-32s ────────────────────────────────────────────────────────
print("=" * 65)
print("Training FCN-32s")
print("=" * 65)

torch.manual_seed(SEED)
fcn32s_model = FCN32s(NUM_CLASSES).to(device)
fcn32s_optimizer = torch.optim.Adam(fcn32s_model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()  # Pixel-wise cross-entropy — see Module 5 for derivation

# History tracking
fcn32s_train_losses: list[float] = []
fcn32s_val_losses: list[float] = []
fcn32s_train_ious: list[float] = []
fcn32s_val_ious: list[float] = []
fcn32s_train_accs: list[float] = []
fcn32s_val_accs: list[float] = []

best_fcn32s_val_loss = float("inf")
best_fcn32s_state: dict = {}

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc, train_iou = train_one_epoch(
        fcn32s_model, train_loader, fcn32s_optimizer, criterion, device
    )
    val_loss, val_acc, val_iou = evaluate(fcn32s_model, val_loader, criterion, device)
    elapsed = time.time() - epoch_start

    fcn32s_train_losses.append(train_loss)
    fcn32s_val_losses.append(val_loss)
    fcn32s_train_ious.append(train_iou)
    fcn32s_val_ious.append(val_iou)
    fcn32s_train_accs.append(train_acc)
    fcn32s_val_accs.append(val_acc)

    if val_loss < best_fcn32s_val_loss:
        best_fcn32s_val_loss = val_loss
        best_fcn32s_state = {k: v.clone() for k, v in fcn32s_model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | Val mIoU: {val_iou:.4f} | Time: {elapsed:.1f}s"
    )

fcn32s_model.load_state_dict(best_fcn32s_state)
print(f"\nBest FCN-32s Val Loss: {best_fcn32s_val_loss:.4f}")

In [ ]:
# ── Training: FCN-8s ─────────────────────────────────────────────────────────
print("=" * 65)
print("Training FCN-8s")
print("=" * 65)

torch.manual_seed(SEED)
fcn8s_model = FCN8s(NUM_CLASSES).to(device)
fcn8s_optimizer = torch.optim.Adam(fcn8s_model.parameters(), lr=LEARNING_RATE)

fcn8s_train_losses: list[float] = []
fcn8s_val_losses: list[float] = []
fcn8s_train_ious: list[float] = []
fcn8s_val_ious: list[float] = []
fcn8s_train_accs: list[float] = []
fcn8s_val_accs: list[float] = []

best_fcn8s_val_loss = float("inf")
best_fcn8s_state: dict = {}

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc, train_iou = train_one_epoch(
        fcn8s_model, train_loader, fcn8s_optimizer, criterion, device
    )
    val_loss, val_acc, val_iou = evaluate(fcn8s_model, val_loader, criterion, device)
    elapsed = time.time() - epoch_start

    fcn8s_train_losses.append(train_loss)
    fcn8s_val_losses.append(val_loss)
    fcn8s_train_ious.append(train_iou)
    fcn8s_val_ious.append(val_iou)
    fcn8s_train_accs.append(train_acc)
    fcn8s_val_accs.append(val_acc)

    if val_loss < best_fcn8s_val_loss:
        best_fcn8s_val_loss = val_loss
        best_fcn8s_state = {k: v.clone() for k, v in fcn8s_model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | Val mIoU: {val_iou:.4f} | Time: {elapsed:.1f}s"
    )

fcn8s_model.load_state_dict(best_fcn8s_state)
print(f"\nBest FCN-8s Val Loss: {best_fcn8s_val_loss:.4f}")

In [ ]:
# ── Training: U-Net ──────────────────────────────────────────────────────────
print("=" * 65)
print("Training U-Net")
print("=" * 65)

torch.manual_seed(SEED)
unet_model = UNet(NUM_CLASSES).to(device)
unet_optimizer = torch.optim.Adam(unet_model.parameters(), lr=LEARNING_RATE)

unet_train_losses: list[float] = []
unet_val_losses: list[float] = []
unet_train_ious: list[float] = []
unet_val_ious: list[float] = []
unet_train_accs: list[float] = []
unet_val_accs: list[float] = []

best_unet_val_loss = float("inf")
best_unet_state: dict = {}

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc, train_iou = train_one_epoch(
        unet_model, train_loader, unet_optimizer, criterion, device
    )
    val_loss, val_acc, val_iou = evaluate(unet_model, val_loader, criterion, device)
    elapsed = time.time() - epoch_start

    unet_train_losses.append(train_loss)
    unet_val_losses.append(val_loss)
    unet_train_ious.append(train_iou)
    unet_val_ious.append(val_iou)
    unet_train_accs.append(train_acc)
    unet_val_accs.append(val_acc)

    if val_loss < best_unet_val_loss:
        best_unet_val_loss = val_loss
        best_unet_state = {k: v.clone() for k, v in unet_model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | Val mIoU: {val_iou:.4f} | Time: {elapsed:.1f}s"
    )

unet_model.load_state_dict(best_unet_state)
print(f"\nBest U-Net Val Loss: {best_unet_val_loss:.4f}")

In [ ]:
# ── Training curves for all three models ─────────────────────────────────────
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
axes[0].plot(epochs_range, fcn32s_val_losses, "o-", label="FCN-32s", color="#e74c3c")
axes[0].plot(epochs_range, fcn8s_val_losses,  "s-", label="FCN-8s",  color="#3498db")
axes[0].plot(epochs_range, unet_val_losses,   "^-", label="U-Net",   color="#2ecc71")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Loss")
axes[0].set_title("Validation Loss")
axes[0].legend()

# Pixel accuracy curves
axes[1].plot(epochs_range, fcn32s_val_accs, "o-", label="FCN-32s", color="#e74c3c")
axes[1].plot(epochs_range, fcn8s_val_accs,  "s-", label="FCN-8s",  color="#3498db")
axes[1].plot(epochs_range, unet_val_accs,   "^-", label="U-Net",   color="#2ecc71")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Pixel Accuracy")
axes[1].set_title("Validation Pixel Accuracy")
axes[1].legend()

# Mean IoU curves
axes[2].plot(epochs_range, fcn32s_val_ious, "o-", label="FCN-32s", color="#e74c3c")
axes[2].plot(epochs_range, fcn8s_val_ious,  "s-", label="FCN-8s",  color="#3498db")
axes[2].plot(epochs_range, unet_val_ious,   "^-", label="U-Net",   color="#2ecc71")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Mean IoU")
axes[2].set_title("Validation Mean IoU")
axes[2].legend()

plt.suptitle("Training Curves: FCN-32s vs FCN-8s vs U-Net", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 5 — Evaluation & Analysis

With all three models trained, we now analyse them comprehensively:

1. **Quantitative summary:** Final validation metrics table.
2. **Per-class IoU breakdown:** Which classes are each model better at?
3. **Qualitative comparison:** Side-by-side predictions on 4 validation samples.
4. **Confusion matrix:** Where does the best model (U-Net) make errors?
5. **Ablation study:** U-Net with bilinear upsampling vs transposed convolution upsampling.

### 3.2 Quantitative Evaluation

We evaluate both architectures using standard segmentation metrics: pixel accuracy, class-wise IoU, and mean IoU. These metrics capture different aspects of segmentation quality beyond simple loss values.

In [ ]:
# ── Final validation metrics summary ─────────────────────────────────────────
def get_final_metrics(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    model_name: str,
    num_params: int,
) -> dict[str, object]:
    """Evaluate model and return a metrics dictionary.

    Args:
        model: Trained segmentation model.
        dataloader: Evaluation DataLoader.
        criterion: Loss function.
        device: Compute device.
        model_name: Display name for the model.
        num_params: Number of trainable parameters.

    Returns:
        Dictionary with keys: Model, Val Loss, Pixel Acc, Mean IoU, Params.
    """
    val_loss, val_acc, val_miou = evaluate(model, dataloader, criterion, device)
    return {
        "Model": model_name,
        "Val Loss": round(val_loss, 4),
        "Pixel Acc": f"{val_acc:.2%}",
        "Mean IoU": round(val_miou, 4),
        "Params": f"{num_params:,}",
    }


summary_rows = [
    get_final_metrics(fcn32s_model, val_loader, criterion, device, "FCN-32s", num_params_fcn32s),
    get_final_metrics(fcn8s_model,  val_loader, criterion, device, "FCN-8s",  num_params_fcn8s),
    get_final_metrics(unet_model,   val_loader, criterion, device, "U-Net",   num_params_unet),
]
summary_df = pd.DataFrame(summary_rows)
print("Final Validation Metrics:")
print(summary_df.to_string(index=False))

In [ ]:
# ── Per-class IoU breakdown ───────────────────────────────────────────────────
def compute_dataset_per_class_iou(
    model: nn.Module,
    dataloader: DataLoader,
    num_classes: int,
    device: torch.device,
) -> np.ndarray:
    """Compute per-class IoU over an entire dataset.

    Accumulates per-class intersection and union counts across all batches,
    then computes IoU from the accumulated totals (more accurate than
    averaging per-batch IoU).

    Args:
        model: Trained segmentation model.
        dataloader: DataLoader over the evaluation dataset.
        num_classes: Total number of classes.
        device: Compute device.

    Returns:
        Float array of shape (num_classes,) with per-class IoU values.
        Classes absent from the dataset have IoU = NaN.
    """
    model.eval()
    intersections = np.zeros(num_classes, dtype=np.float64)
    unions = np.zeros(num_classes, dtype=np.float64)
    gt_counts = np.zeros(num_classes, dtype=np.int64)

    with torch.no_grad():
        for batch_images, batch_masks in dataloader:
            batch_images = batch_images.to(device)
            logits = model(batch_images)
            pred_masks = logits.argmax(dim=1).cpu()  # (B, H, W)

            pred_flat = pred_masks.reshape(-1).numpy()
            gt_flat = batch_masks.reshape(-1).numpy()

            for cls in range(num_classes):
                pred_cls = pred_flat == cls
                gt_cls = gt_flat == cls
                gt_counts[cls] += gt_cls.sum()
                intersections[cls] += (pred_cls & gt_cls).sum()
                unions[cls] += (pred_cls | gt_cls).sum()

    iou_per_class = np.full(num_classes, float("nan"))
    for cls in range(num_classes):
        if gt_counts[cls] > 0:
            iou_per_class[cls] = intersections[cls] / (unions[cls] + 1e-8)
    return iou_per_class


iou_fcn32s = compute_dataset_per_class_iou(fcn32s_model, val_loader, NUM_CLASSES, device)
iou_fcn8s  = compute_dataset_per_class_iou(fcn8s_model,  val_loader, NUM_CLASSES, device)
iou_unet   = compute_dataset_per_class_iou(unet_model,   val_loader, NUM_CLASSES, device)

# Build per-class IoU table
iou_table_data: dict[str, list] = {"Class": CLASS_NAMES}
for model_label, iou_vals in [("FCN-32s", iou_fcn32s), ("FCN-8s", iou_fcn8s), ("U-Net", iou_unet)]:
    iou_table_data[model_label] = [
        f"{v:.4f}" if not math.isnan(v) else "N/A" for v in iou_vals
    ]

iou_df = pd.DataFrame(iou_table_data)
print("Per-Class IoU Breakdown:")
print(iou_df.to_string(index=False))

# Bar chart comparison
x = np.arange(NUM_CLASSES)
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, iou_fcn32s, width, label="FCN-32s", color="#e74c3c", alpha=0.85)
bars2 = ax.bar(x,          iou_fcn8s,  width, label="FCN-8s",  color="#3498db", alpha=0.85)
bars3 = ax.bar(x + width,  iou_unet,   width, label="U-Net",   color="#2ecc71", alpha=0.85)
ax.set_xlabel("Class")
ax.set_ylabel("IoU")
ax.set_title("Per-Class IoU: FCN-32s vs FCN-8s vs U-Net")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.1)
ax.legend()
ax.axhline(y=1.0, color="grey", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Qualitative comparison: 4 samples ────────────────────────────────────────
def predict_mask(
    model: nn.Module,
    image: torch.Tensor,
    device: torch.device,
) -> torch.Tensor:
    """Run inference on a single image and return the predicted mask.

    Args:
        model: Trained segmentation model.
        image: Image tensor of shape (3, H, W).
        device: Compute device.

    Returns:
        Predicted mask of shape (H, W) with long class indices.
    """
    model.eval()
    with torch.no_grad():
        img_batch = image.unsqueeze(0).to(device)  # (1, 3, H, W)
        logits = model(img_batch)                  # (1, C, H, W)
        pred = logits.argmax(dim=1).squeeze(0)     # (H, W)
    return pred.cpu()


num_qual_samples = 4
qual_indices = [5, 12, 25, 42]  # Fixed indices for reproducibility

fig, axes = plt.subplots(num_qual_samples, 5, figsize=(18, num_qual_samples * 3.2))
col_titles = ["Image", "GT Mask", "FCN-32s", "FCN-8s", "U-Net"]

for col_title, ax in zip(col_titles, axes[0]):
    ax.set_title(col_title, fontsize=11, fontweight="bold")

for row_idx, sample_idx in enumerate(qual_indices):
    img, gt_mask = val_dataset[sample_idx]
    pred_fcn32s = predict_mask(fcn32s_model, img, device)
    pred_fcn8s  = predict_mask(fcn8s_model,  img, device)
    pred_unet   = predict_mask(unet_model,   img, device)

    # Column 0: RGB image
    axes[row_idx, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[row_idx, 0].set_ylabel(f"Sample {sample_idx}", fontsize=9)
    axes[row_idx, 0].set_xticks([])
    axes[row_idx, 0].set_yticks([])

    # Columns 1–4: masks
    for col_idx, mask_data in enumerate([gt_mask, pred_fcn32s, pred_fcn8s, pred_unet]):
        axes[row_idx, col_idx + 1].imshow(
            mask_data.numpy(), cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1
        )
        # Show per-sample mIoU for predictions
        if col_idx > 0:
            miou = compute_mean_iou(mask_data, gt_mask, NUM_CLASSES)
            axes[row_idx, col_idx + 1].set_xlabel(f"mIoU={miou:.3f}", fontsize=8)
        axes[row_idx, col_idx + 1].set_xticks([])
        axes[row_idx, col_idx + 1].set_yticks([])

legend_patches = [
    mpatches.Patch(facecolor=PALETTE[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)
]
fig.legend(handles=legend_patches, loc="lower center", ncol=NUM_CLASSES, fontsize=10)
fig.suptitle("Qualitative Comparison: Image | GT | FCN-32s | FCN-8s | U-Net", fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 0.98])
plt.show()

In [ ]:
# ── Confusion matrix on validation set (U-Net) ────────────────────────────────
def compute_confusion_matrix(
    model: nn.Module,
    dataloader: DataLoader,
    num_classes: int,
    device: torch.device,
) -> np.ndarray:
    """Compute pixel-level confusion matrix over the full dataset.

    Rows = ground truth class, Columns = predicted class.

    Args:
        model: Trained segmentation model.
        dataloader: DataLoader to evaluate.
        num_classes: Number of classes C.
        device: Compute device.

    Returns:
        Int64 array of shape (C, C) — confusion matrix.
    """
    model.eval()
    conf_matrix = np.zeros((num_classes, num_classes), dtype=np.int64)

    with torch.no_grad():
        for batch_images, batch_masks in dataloader:
            batch_images = batch_images.to(device)
            logits = model(batch_images)
            pred_masks = logits.argmax(dim=1).cpu().numpy().flatten()
            gt_flat = batch_masks.numpy().flatten()

            for true_cls, pred_cls in zip(gt_flat, pred_masks):
                conf_matrix[true_cls, pred_cls] += 1

    return conf_matrix


unet_conf = compute_confusion_matrix(unet_model, val_loader, NUM_CLASSES, device)

# Normalise by row (true class) to get recall
row_sums = unet_conf.sum(axis=1, keepdims=True)
unet_conf_norm = unet_conf / (row_sums + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
im0 = axes[0].imshow(unet_conf, cmap="Blues")
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_xticklabels(CLASS_NAMES, rotation=30)
axes[0].set_yticklabels(CLASS_NAMES)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Ground Truth")
axes[0].set_title("U-Net Confusion Matrix (pixel counts)")
for row_i in range(NUM_CLASSES):
    for col_j in range(NUM_CLASSES):
        axes[0].text(
            col_j, row_i, f"{unet_conf[row_i, col_j]:,}",
            ha="center", va="center", fontsize=9,
            color="white" if unet_conf[row_i, col_j] > unet_conf.max() * 0.5 else "black"
        )
plt.colorbar(im0, ax=axes[0])

# Normalised (recall per class)
im1 = axes[1].imshow(unet_conf_norm, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_yticks(range(NUM_CLASSES))
axes[1].set_xticklabels(CLASS_NAMES, rotation=30)
axes[1].set_yticklabels(CLASS_NAMES)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Ground Truth")
axes[1].set_title("U-Net Confusion Matrix (row-normalised recall)")
for row_i in range(NUM_CLASSES):
    for col_j in range(NUM_CLASSES):
        axes[1].text(
            col_j, row_i, f"{unet_conf_norm[row_i, col_j]:.2f}",
            ha="center", va="center", fontsize=9,
            color="white" if unet_conf_norm[row_i, col_j] > 0.5 else "black"
        )
plt.colorbar(im1, ax=axes[1])

plt.suptitle("U-Net Pixel-Level Confusion Analysis (Validation Set)", fontsize=12)
plt.tight_layout()
plt.show()

### 5.4 Ablation: U-Net Bilinear vs Transposed Conv Upsampling

Our U-Net decoder uses **bilinear upsampling** (no learnable parameters in the upsampling step, followed by DoubleConv for feature refinement). An alternative is **transposed convolution upsampling** — learnable but prone to checkerboard artifacts.

We now build a `UNetTransposedDec` variant that replaces `nn.Upsample` with `nn.ConvTranspose2d(stride=2)` in the decoder and trains it for the same number of epochs, comparing final mIoU and convergence speed.

### Part 4 — Comparison & Analysis

We compare FCN and U-Net architectures on the same synthetic segmentation task, analysing how skip connections affect boundary quality, convergence speed, and mean IoU performance.

In [ ]:
class UNetTransposedDecoder(nn.Module):
    """U-Net decoder using transposed convolution instead of bilinear upsampling.

    This is an ablation variant of UNetDecoder that replaces nn.Upsample
    with nn.ConvTranspose2d(stride=2) for the upsampling step at each level.
    Demonstrates the impact of the upsampling method on segmentation quality.

    Attributes:
        tconv4: Transposed conv 2× upsample for level 4 (256 → 256 channels).
        up4: DoubleConv after concat at level 4 (512 → 256).
        tconv3: Transposed conv 2× upsample for level 3 (256 → 256 channels).
        up3: DoubleConv after concat at level 3 (384 → 128).
        tconv2: Transposed conv 2× upsample for level 2 (128 → 128 channels).
        up2: DoubleConv after concat at level 2 (192 → 64).
        tconv1: Transposed conv 2× upsample for level 1 (64 → 64 channels).
        up1: DoubleConv after concat at level 1 (96 → 32).
    """

    def __init__(self) -> None:
        """Initialise transposed-conv decoder with four upsampling stages."""
        super().__init__()
        self.tconv4 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.up4 = DoubleConv(256 + 256, 256)
        self.tconv3 = nn.ConvTranspose2d(256, 256, kernel_size=2, stride=2)
        self.up3 = DoubleConv(256 + 128, 128)
        self.tconv2 = nn.ConvTranspose2d(128, 128, kernel_size=2, stride=2)
        self.up2 = DoubleConv(128 + 64, 64)
        self.tconv1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.up1 = DoubleConv(64 + 32, 32)

    def forward(
        self,
        skip1: torch.Tensor,
        skip2: torch.Tensor,
        skip3: torch.Tensor,
        skip4: torch.Tensor,
        bottleneck: torch.Tensor,
    ) -> torch.Tensor:
        """Run decoder with transposed conv upsampling and skip concatenation.

        Args:
            skip1: Encoder level-1 features (B, 32,  H,    W).
            skip2: Encoder level-2 features (B, 64,  H/2,  W/2).
            skip3: Encoder level-3 features (B, 128, H/4,  W/4).
            skip4: Encoder level-4 features (B, 256, H/8,  W/8).
            bottleneck: Deepest features    (B, 256, H/16, W/16).

        Returns:
            Decoded feature tensor of shape (B, 32, H, W).
        """
        x = self.tconv4(bottleneck)              # (B, 256, H/8, W/8)
        x = torch.cat([x, skip4], dim=1)         # (B, 512, H/8, W/8)
        x = self.up4(x)                          # (B, 256, H/8, W/8)

        x = self.tconv3(x)                       # (B, 256, H/4, W/4)
        x = torch.cat([x, skip3], dim=1)         # (B, 384, H/4, W/4)
        x = self.up3(x)                          # (B, 128, H/4, W/4)

        x = self.tconv2(x)                       # (B, 128, H/2, W/2)
        x = torch.cat([x, skip2], dim=1)         # (B, 192, H/2, W/2)
        x = self.up2(x)                          # (B, 64,  H/2, W/2)

        x = self.tconv1(x)                       # (B, 64,  H, W)
        x = torch.cat([x, skip1], dim=1)         # (B, 96,  H, W)
        x = self.up1(x)                          # (B, 32,  H, W)
        return x


class UNetTransposed(nn.Module):
    """U-Net using transposed convolution upsampling in the decoder.

    Ablation variant of UNet where the bilinear nn.Upsample in the decoder
    is replaced with nn.ConvTranspose2d(stride=2). All other components
    (encoder, DoubleConv blocks, skip concatenation) remain identical.

    Attributes:
        encoder: UNetEncoder (shared architecture with standard U-Net).
        decoder: UNetTransposedDecoder with transposed conv upsampling.
        head: Final 1×1 conv projecting 32 channels to num_classes.
    """

    def __init__(self, num_classes: int) -> None:
        """Initialise transposed-conv U-Net.

        Args:
            num_classes: Number of output segmentation classes.
        """
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetTransposedDecoder()
        self.head = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: image → per-pixel class logits.

        Args:
            x: Input tensor of shape (B, 3, H, W).

        Returns:
            Logit tensor of shape (B, num_classes, H, W).
        """
        skip1, skip2, skip3, skip4, bottleneck = self.encoder(x)
        decoded = self.decoder(skip1, skip2, skip3, skip4, bottleneck)
        return self.head(decoded)


# Verify output shape
unet_t_check = UNetTransposed(NUM_CLASSES).to(device)
unet_t_out = unet_t_check(dummy_input)
assert unet_t_out.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), "Shape mismatch!"
num_params_unet_t = sum(p.numel() for p in unet_t_check.parameters() if p.requires_grad)
print(f"U-Net (transposed) parameters: {num_params_unet_t:,}")

In [ ]:
# ── Train U-Net (transposed conv) for ablation ────────────────────────────────
print("=" * 65)
print("Ablation: Training U-Net (Transposed Conv Decoder)")
print("=" * 65)

torch.manual_seed(SEED)
unet_t_model = UNetTransposed(NUM_CLASSES).to(device)
unet_t_optimizer = torch.optim.Adam(unet_t_model.parameters(), lr=LEARNING_RATE)

unet_t_val_losses: list[float] = []
unet_t_val_ious: list[float] = []
unet_t_val_accs: list[float] = []

best_unet_t_val_loss = float("inf")
best_unet_t_state: dict = {}

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    train_loss, train_acc, train_iou = train_one_epoch(
        unet_t_model, train_loader, unet_t_optimizer, criterion, device
    )
    val_loss, val_acc, val_iou = evaluate(unet_t_model, val_loader, criterion, device)
    elapsed = time.time() - epoch_start

    unet_t_val_losses.append(val_loss)
    unet_t_val_ious.append(val_iou)
    unet_t_val_accs.append(val_acc)

    if val_loss < best_unet_t_val_loss:
        best_unet_t_val_loss = val_loss
        best_unet_t_state = {k: v.clone() for k, v in unet_t_model.state_dict().items()}

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2%} | Val mIoU: {val_iou:.4f} | Time: {elapsed:.1f}s"
    )

unet_t_model.load_state_dict(best_unet_t_state)
print(f"\nBest U-Net (Transposed) Val Loss: {best_unet_t_val_loss:.4f}")

In [ ]:
# ── Ablation results table ────────────────────────────────────────────────────
ablation_rows = [
    get_final_metrics(unet_model,   val_loader, criterion, device, "U-Net (Bilinear)",    num_params_unet),
    get_final_metrics(unet_t_model, val_loader, criterion, device, "U-Net (Transposed)",  num_params_unet_t),
]
ablation_df = pd.DataFrame(ablation_rows)
print("Ablation Study: Upsampling Method")
print(ablation_df.to_string(index=False))

# Convergence comparison plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, unet_val_losses,   "^-", label="Bilinear",    color="#2ecc71")
axes[0].plot(epochs_range, unet_t_val_losses, "D-", label="Transposed",  color="#9b59b6")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Loss")
axes[0].set_title("Ablation: Val Loss")
axes[0].legend()

axes[1].plot(epochs_range, unet_val_ious,   "^-", label="Bilinear",    color="#2ecc71")
axes[1].plot(epochs_range, unet_t_val_ious, "D-", label="Transposed",  color="#9b59b6")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean IoU")
axes[1].set_title("Ablation: Val Mean IoU")
axes[1].legend()

plt.suptitle("U-Net Ablation: Bilinear vs Transposed Conv Upsampling", fontsize=12)
plt.tight_layout()
plt.show()

# Qualitative: compare U-Net bilinear vs transposed on one sample
sample_img_abl, sample_gt_abl = val_dataset[10]
pred_bilinear = predict_mask(unet_model,   sample_img_abl, device)
pred_transpos = predict_mask(unet_t_model, sample_img_abl, device)

fig_abl, abl_axes = plt.subplots(1, 4, figsize=(14, 3.5))
abl_axes[0].imshow(sample_img_abl.permute(1, 2, 0).numpy())
abl_axes[0].set_title("Image")
abl_axes[0].axis("off")
abl_axes[1].imshow(sample_gt_abl.numpy(), cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
abl_axes[1].set_title("GT Mask")
abl_axes[1].axis("off")
miou_bil = compute_mean_iou(pred_bilinear, sample_gt_abl, NUM_CLASSES)
abl_axes[2].imshow(pred_bilinear.numpy(), cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
abl_axes[2].set_title(f"Bilinear\nmIoU={miou_bil:.3f}")
abl_axes[2].axis("off")
miou_tra = compute_mean_iou(pred_transpos, sample_gt_abl, NUM_CLASSES)
abl_axes[3].imshow(pred_transpos.numpy(), cmap=SEG_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
abl_axes[3].set_title(f"Transposed\nmIoU={miou_tra:.3f}")
abl_axes[3].axis("off")
plt.suptitle("Upsampling Ablation: Sample Prediction", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final comprehensive comparison table ──────────────────────────────────────
all_model_rows = [
    get_final_metrics(fcn32s_model,  val_loader, criterion, device, "FCN-32s",           num_params_fcn32s),
    get_final_metrics(fcn8s_model,   val_loader, criterion, device, "FCN-8s",            num_params_fcn8s),
    get_final_metrics(unet_model,    val_loader, criterion, device, "U-Net (Bilinear)",  num_params_unet),
    get_final_metrics(unet_t_model,  val_loader, criterion, device, "U-Net (Transposed)", num_params_unet_t),
]
all_models_df = pd.DataFrame(all_model_rows)
print("\n" + "=" * 65)
print("Comprehensive Model Comparison (Validation Set)")
print("=" * 65)
print(all_models_df.to_string(index=False))

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **FCN-32s vs FCN-8s:** Skip connections dramatically improve segmentation sharpness. By fusing coarse (semantic) deep features with finer (spatial) shallow features, FCN-8s recovers detail that FCN-32s blurs away in a single large transposed convolution. This is visible both in IoU scores and qualitative predictions.

2. **U-Net's symmetric architecture:** U-Net extends the skip idea fully — at every resolution level, the complete feature map (not just a projected score) is concatenated into the decoder. This richer information flow gives the decoder full access to both semantic context (deep) and spatial precision (shallow), enabling sharp boundary predictions with relatively few parameters.

3. **Upsampling method matters:** Bilinear interpolation is parameter-free and smooth; transposed convolutions are learnable but prone to checkerboard artifacts if `kernel_size % stride ≠ 0`. The standard U-Net recipe (bilinear upsample + DoubleConv) avoids artifacts while still adapting features through learned convolutions after upsampling.

4. **IoU vs pixel accuracy:** Pixel accuracy can be misleading when the background dominates (as it does here). Mean IoU is more informative because it treats each class equally — a model that perfectly predicts background but ignores small foreground shapes will score high on accuracy but low on mIoU.

5. **Pixel-wise cross-entropy:** `nn.CrossEntropyLoss` applied to $(B, C, H, W)$ logits and $(B, H, W)$ targets seamlessly extends standard classification loss to dense prediction — every pixel contributes an independent term to the loss, making implementation straightforward.

### What's Next

→ **09-04 (Instance Segmentation Concepts)** extends dense prediction by adding instance-level discrimination: RoI Align from scratch and a mask head that predicts per-instance binary masks, building directly on the encoder-decoder ideas from this notebook.

### Going Further

- **Long et al. (2015):** "Fully Convolutional Networks for Semantic Segmentation" — the original FCN paper.
- **Ronneberger et al. (2015):** "U-Net: Convolutional Networks for Biomedical Image Segmentation" — the original U-Net paper with full architecture diagrams.
- **DeepLab / ASPP:** Atrous (dilated) convolutions for multi-scale context without spatial resolution loss — an alternative to encoder-decoder architectures.
- **Transformer-based segmentation:** SegFormer, Mask2Former — applying attention mechanisms to dense prediction (builds on ViT introduced in 09-05).